# 2D3V 场校正测试
均匀场的空间导数为零，因此磁场保持不变，零速度和零压力时电场为零。

In [ ]:
import os, sys
sys.path.insert(0, os.environ['BORIS_MODULE_DIR'])
import boris_cpp as boris

grid = boris.PeriodicGrid(boris.Vec3(0, 0, 0), boris.Vec3(4, 4, 0), (4, 4, 1))
data = boris.CorrectionInput()
data.magnetic = [boris.Vec3(0, 0, 2) for _ in range(grid.size())]
data.electric_half = [boris.Vec3(0, 0, 0) for _ in range(grid.size())]
data.density = [1.0 for _ in range(grid.size())]
data.bulk_velocity = [boris.Vec3(0, 0, 0) for _ in range(grid.size())]
data.electron_pressure = [boris.Vec3(0, 0, 0) for _ in range(grid.size())]

result = boris.correct_fields(grid, data, 0.1, 0.0, 0.0)
assert len(result.magnetic) == grid.size()
assert len(result.electric) == grid.size()
assert all(b.z == 2 for b in result.magnetic)
assert all(e.x == 0 and e.y == 0 and e.z == 0 for e in result.electric)
print('均匀场校正通过。')

# 检查二维 curl：只随 x 变化的 Fz 会产生 y 方向 curl。
field = [boris.Vec3(0, 0, [0, 1, 0, -1][i // 4]) for i in range(grid.size())]
curl = boris.curl_periodic(grid, field)
assert abs(curl[0].y + 1.0) < 1e-12
assert abs(curl[8].y - 1.0) < 1e-12

# 法拉第方程：B 新 = B 旧 - dt * curl(E)。
magnetic_next = boris.advance_magnetic_faraday(
    grid, [boris.Vec3(0, 0, 0) for _ in range(grid.size())], field, 0.1
)
assert abs(magnetic_next[0].y - 0.1) < 1e-12
assert abs(magnetic_next[8].y + 0.1) < 1e-12

# Laplacian 只沿 x、y 计算；这个场只沿 x 变化。
lap = boris.laplacian_periodic(grid, field)
assert abs(lap[4].z + 2.0) < 1e-12
print('二维 curl 和 Laplacian 通过。')